In [ ]:
import osmnx as ox
import pandas as pd
import os
import networkx as nx
import geopy.distance
import numpy as np
import geopandas as gpd

In [ ]:
#4 representative clusters
ox.settings.use_cache = True
clusters = {
    "urban_honolulu":{ "cluster_coords":(21.304547, -157.855676), },  
    "suburban_mililani": {"cluster_coords":(21.4508308, -158.0095783),},
    "rural_waianae": { "cluster_coords":(21.4454875, -158.1878239),},
    "remote_ocean_view":{"cluster_coords":(19.1033522, -155.785762),}                          
}

In [ ]:
cai_approved=pd.read_excel("../data/raw/cai_approved.xlsx")

In [ ]:
for name, data in clusters.items():
    filepath = f"../data/raw/{name}_network.graphml"
    if os.path.exists(filepath):
        G=ox.load_graphml(filepath)
        print(f"{name} already exists")
    else:
        G = ox.graph_from_point(data["cluster_coords"], dist=8000, network_type="drive")
        ox.save_graphml(G,filepath)
        print(f"{name} saved.")  

In [ ]:
#take closest 3 cais, sort from smallest to largest
cluster_cais={}
for name, data in clusters.items():
    cluster_center = data["cluster_coords"]
    df_temp = cai_approved.copy()
    df_temp["cluster_distance"] = df_temp.apply(lambda row: geopy.distance.geodesic((row["latitude"], row["longitude"]),cluster_center).km,
    axis=1)
    cluster_cais[name]=df_temp.nsmallest(3,"cluster_distance")

In [ ]:
#distance between every road node to the road node from the nearest cai
for name, data in clusters.items():
    filepath = f"../data/raw/{name}_network.graphml"
    G = ox.load_graphml(filepath)
    target_df=cluster_cais[name]
    cai_nodes = [ox.nearest_nodes(G, X=row['longitude'],Y=row['latitude']) 
    for _, row in target_df.iterrows()]
    distances = nx.multi_source_dijkstra_path_length(G, list(set(cai_nodes)), weight='length')
    nx.set_node_attributes(G, distances, "dist_to_cai_proxy")
    ox.save_graphml(G, filepath=f"../data/raw/{name}_network_with_distance.graphml")
    print(f"saved {name} network with distance.")   

In [ ]:
#summary stats for calculated proxy distances
for name in clusters:
    G_test = ox.load_graphml(f"../data/raw/{name}_network_with_distance.graphml")
    distances = [
        float(data["dist_to_cai_proxy"])
        for node, data in G_test.nodes(data=True)
        if "dist_to_cai_proxy" in data
    ]
    print(
        name,
        "mean:", np.mean(distances),
        "median:", np.median(distances),
        "max:", np.max(distances)
    )

The cell below contains pre-calculated terrain statistics, followed by calculations for a single cluster, hence the commented out code per cluster in the cell below.

In [ ]:
terrain_scores = {
    "urban_honolulu": {"terrain_p75": 0.0435},
    "suburban_mililani": {"terrain_p75": 0.0328},
    "rural_waianae": {"terrain_p75": 0.0242},
    "remote_ocean_view": { "terrain_p75": 0.0845}
}
#name="urban_honolulu"
#name="suburban_mililani"
#name="rural_waianae"
name="remote_ocean_view"
G=ox.load_graphml(f"../data/raw/{name}_network_with_distance.graphml")

In [ ]:
#add elevation  
G= ox.elevation.add_node_elevations_raster( G, "../data/raw/n20w1562023.tif")
#calcuate road slope
G = ox.elevation.add_edge_grades(G)
grades = [
    data["grade_abs"]
    for _, _, data in G.edges(data=True)
    if "grade_abs" in data
]
terrain_scores[name] = {
    "median": np.percentile(grades, 50),
    "p75": np.percentile(grades, 75),
    "p90": np.percentile(grades, 90)
}
ox.save_graphml(G, f"../data/raw/{name}_network_with_terrain.graphml")
print(terrain_scores)

In [ ]:
housing = pd.read_csv("../data/raw/acs_data/housing.csv", skiprows=1)
housing.head()

In [ ]:
#removing duplicate header row and rename columns
housing = housing[housing["Geography"] != "Geography"]
housing = housing.rename(columns={
    "Geography": "GEO_ID",
    "Estimate!!Total": "housing"
})
#takes out prefix from every geo id
housing["GEOID"] = housing["GEO_ID"].str.replace(
    "1500000US",
    "",
    regex=False
)
housing["GEOID"].head()

In [ ]:
census = gpd.read_file( "../data/raw/census data/tl_2021_15_bg.shp")
#merges geo id and housing count with census dataset
census = census.merge(
    housing[["GEOID", "housing"]],
    on="GEOID",
    how="left"
)
#converts housing to a float/numeric data type
census["housing"] = pd.to_numeric(
    census["housing"],
    errors="coerce"
)
print(census[["GEOID", "housing"]].head())

In [ ]:
#housing density value for every census block group in Hawaii
census["area_km2"] = census["ALAND"] / 1e6
census["housing_density"] = census["housing"] / census["area_km2"]
census["housing_density"].describe()

In [ ]:
census.head()

In [ ]:
#transfer density values to network nodes
def add_density_to_nodes(G, census):
    nodes, edges = ox.graph_to_gdfs(G)
    census_proj = census.to_crs(nodes.crs)
    
    nodes = gpd.sjoin(
        nodes,
        census_proj[["geometry", "housing_density"]],
        how="left",
        predicate="within"
    )
    
    nodes["housing_density"] = nodes["housing_density"].fillna(0)
    return nodes, edges

In [ ]:
#takes average of start node u and end node v
def add_density_to_edges(nodes, edges):
    edges = edges.reset_index()
    edges["housing_density"] = (
        edges["u"].map(nodes["housing_density"]) +
        edges["v"].map(nodes["housing_density"])
    ) / 2
    return edges

In [ ]:
nodes_dict = {}
edges_dict = {}

for name in clusters:
    print("PROCESSING:", name)
    
    G_terrain = ox.load_graphml(f"../data/raw/{name}_network_with_terrain.graphml")
    nodes, edges = ox.graph_to_gdfs(G_terrain)
    
    G_distance = ox.load_graphml(f"../data/raw/{name}_network_with_distance.graphml")
    nodes_distance, _ = ox.graph_to_gdfs(G_distance)
    
    #calculate grade per node from edges 
    edges_reset = edges.reset_index()
    grade_per_node = pd.concat([
        edges_reset[["u", "grade_abs"]].rename(columns={"u": "osmid"}),
        edges_reset[["v", "grade_abs"]].rename(columns={"v": "osmid"})
    ]).groupby("osmid")["grade_abs"].mean()
    nodes["grade_abs"] = nodes.index.map(grade_per_node)
    
    #adding housing density
    nodes_with_density, _ = add_density_to_nodes(G_terrain, census)
    nodes["housing_density"] = nodes_with_density["housing_density"]
    edges = add_density_to_edges(nodes, edges)
    
    #adding CAI proxy distance
    nodes["dist_to_cai_proxy"] = pd.to_numeric(nodes_distance["dist_to_cai_proxy"], errors="coerce")
    edges["dist_to_cai_proxy"] = (
        edges["u"].map(nodes["dist_to_cai_proxy"])+
        edges["v"].map(nodes["dist_to_cai_proxy"])
    )/2
    
    nodes_dict[name] = nodes
    edges_dict[name] = edges
    
    display(nodes[["grade", "housing_density", "dist_to_cai_proxy"]].head())
    nodes.to_file(f"../data/processed/{name}_nodes_with_features.gpkg", driver="GPKG")
    edges.to_file(f"../data/processed/{name}_edges_with_features.gpkg", driver="GPKG")